# 00 — Data Preparation
**Purpose:** Merge raw CSVs (water quality, landsat, terraclimate), clean column names, handle dtypes, verify no data leakage.

**Input:** Raw CSVs from Kaggle Dataset (`ey-water-quality-data/`)

**Output:** `train_base.parquet`, `val_base.parquet`

**Figures:**
- 📍 Station location map (train vs validation)
- 📊 Target distributions
- 🔍 Missing values heatmap
- 📈 Samples per station

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# === Plotting Style ===
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'figure.dpi': 120,
})

SEED = 42
np.random.seed(SEED)

# Kaggle input path (adjust if running locally)
INPUT_DIR = '/kaggle/input/ey-water-quality-data'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Input directory: {INPUT_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

## 1. Load Raw Data

In [ ]:
# === Target dataset ===
wq_train = pd.read_csv(f'{INPUT_DIR}/water_quality_training_dataset.csv')
print(f'Water Quality Training: {wq_train.shape}')
display(wq_train.head())
display(wq_train.dtypes)

In [ ]:
# === Feature datasets ===
landsat_train = pd.read_csv(f'{INPUT_DIR}/landsat_features_training.csv')
landsat_val = pd.read_csv(f'{INPUT_DIR}/landsat_features_validation.csv')
terra_train = pd.read_csv(f'{INPUT_DIR}/terraclimate_features_training.csv')
terra_val = pd.read_csv(f'{INPUT_DIR}/terraclimate_features_validation.csv')
submission = pd.read_csv(f'{INPUT_DIR}/submission_template.csv')

print(f'Landsat Train: {landsat_train.shape}, Val: {landsat_val.shape}')
print(f'TerraClimate Train: {terra_train.shape}, Val: {terra_val.shape}')
print(f'Submission Template: {submission.shape}')

## 2. Inspect & Clean Column Names

In [ ]:
print('=== Water Quality Columns ===')
print(wq_train.columns.tolist())
print('\n=== Landsat Columns ===')
print(landsat_train.columns.tolist())
print('\n=== TerraClimate Columns ===')
print(terra_train.columns.tolist())
print('\n=== Submission Columns ===')
print(submission.columns.tolist())

In [ ]:
def clean_column_names(df):
    """Standardize column names: strip whitespace."""
    df.columns = df.columns.str.strip()
    return df

wq_train = clean_column_names(wq_train)
landsat_train = clean_column_names(landsat_train)
landsat_val = clean_column_names(landsat_val)
terra_train = clean_column_names(terra_train)
terra_val = clean_column_names(terra_val)

print('Cleaned columns:')
print(wq_train.columns.tolist())

## 3. Identify Merge Keys & Target Columns

In [ ]:
# Identify the merge keys common across datasets
wq_cols = set(wq_train.columns)
ls_cols = set(landsat_train.columns)
tc_cols = set(terra_train.columns)

common_all = wq_cols & ls_cols & tc_cols
print(f'Common to all 3: {common_all}')
print(f'\nWQ ∩ Landsat: {wq_cols & ls_cols}')
print(f'WQ ∩ TerraClimate: {wq_cols & tc_cols}')

In [ ]:
# Auto-detect target columns
TARGET_COLS = []
for col in wq_train.columns:
    col_lower = col.lower()
    if any(keyword in col_lower for keyword in ['alkalinity', 'conductance', 'phosphorus']):
        TARGET_COLS.append(col)

print(f'Target columns detected: {TARGET_COLS}')

# Verify targets
for col in TARGET_COLS:
    print(f'  {col}: dtype={wq_train[col].dtype}, nulls={wq_train[col].isnull().sum()}, '
          f'range=[{wq_train[col].min():.2f}, {wq_train[col].max():.2f}]')

## 4. Parse Dates & Merge

In [ ]:
# Parse date columns
date_col_candidates = [c for c in wq_train.columns if 'date' in c.lower()]
DATE_COL = date_col_candidates[0] if date_col_candidates else None
print(f'Using date column: {DATE_COL}')

if DATE_COL:
    for df in [wq_train, landsat_train, landsat_val, terra_train, terra_val]:
        if DATE_COL in df.columns:
            df[DATE_COL] = pd.to_datetime(df[DATE_COL], format='mixed', dayfirst=False)
    print(f'Date range (WQ): {wq_train[DATE_COL].min()} to {wq_train[DATE_COL].max()}')

In [ ]:
# Merge datasets
MERGE_KEYS = sorted(list(common_all))
print(f'Merge keys: {MERGE_KEYS}')

train_merged = wq_train.merge(landsat_train, on=MERGE_KEYS, how='left', suffixes=('', '_landsat'))
print(f'After Landsat merge: {train_merged.shape}')

train_merged = train_merged.merge(terra_train, on=MERGE_KEYS, how='left', suffixes=('', '_terra'))
print(f'After TerraClimate merge: {train_merged.shape}')

# Validation base (no targets)
val_merged = landsat_val.merge(terra_val, on=MERGE_KEYS, how='left', suffixes=('', '_terra'))
print(f'Validation merged: {val_merged.shape}')

## 5. Data Quality Checks

In [ ]:
# Detect station and coordinate columns
station_col_candidates = [c for c in train_merged.columns if 'station' in c.lower() or 'gems' in c.lower()]
STATION_COL = station_col_candidates[0] if station_col_candidates else None

lat_candidates = [c for c in train_merged.columns if 'lat' in c.lower()]
lon_candidates = [c for c in train_merged.columns if 'lon' in c.lower()]
LAT_COL = lat_candidates[0] if lat_candidates else None
LON_COL = lon_candidates[0] if lon_candidates else None

print(f'Station column: {STATION_COL}')
print(f'Lat/Lon columns: {LAT_COL}, {LON_COL}')

# === LEAKAGE CHECK ===
if STATION_COL:
    train_stations = set(train_merged[STATION_COL].unique())
    val_stations = set(val_merged[STATION_COL].unique()) if STATION_COL in val_merged.columns else set()
    overlap = train_stations & val_stations
    
    print(f'\nTraining stations: {len(train_stations)}')
    print(f'Validation stations: {len(val_stations)}')
    print(f'Overlapping stations: {len(overlap)}')
    
    if overlap:
        print(f'⚠️ WARNING: Station overlap detected! {overlap}')
    else:
        print('✅ No station overlap — spatial extrapolation confirmed')

---
## 📊 FIGURE 1: Station Location Map (Train vs Validation)

In [ ]:
if LAT_COL and LON_COL:
    # Get unique station locations
    train_locs = train_merged.groupby(STATION_COL).agg({LAT_COL: 'first', LON_COL: 'first'}).reset_index()
    val_locs = val_merged.groupby(STATION_COL).agg({LAT_COL: 'first', LON_COL: 'first'}).reset_index() if STATION_COL in val_merged.columns else val_merged[[LAT_COL, LON_COL]].drop_duplicates()
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # South Africa bounding box for context
    ax.set_xlim(16, 33)
    ax.set_ylim(-35, -22)
    
    # Plot stations
    ax.scatter(train_locs[LON_COL], train_locs[LAT_COL], 
               c='#2196F3', s=80, alpha=0.8, edgecolors='white', linewidths=0.5,
               label=f'Training ({len(train_locs)} stations)', zorder=3)
    ax.scatter(val_locs[LON_COL], val_locs[LAT_COL], 
               c='#FF5722', s=120, marker='*', edgecolors='white', linewidths=0.5,
               label=f'Validation ({len(val_locs)} stations)', zorder=4)
    
    # Major city markers for reference
    cities = {
        'Cape Town': (-33.92, 18.42), 'Johannesburg': (-26.20, 28.04),
        'Durban': (-29.86, 31.02), 'Pretoria': (-25.75, 28.19),
        'Port Elizabeth': (-33.96, 25.60), 'Bloemfontein': (-29.09, 26.16),
    }
    for city, (lat, lon) in cities.items():
        ax.plot(lon, lat, 'k^', markersize=8, zorder=5)
        ax.annotate(city, (lon, lat), textcoords='offset points', xytext=(5, 5),
                   fontsize=8, color='gray')
    
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    ax.set_title('📍 Station Locations: Training vs Validation\n'
                 'Validation stations are in DIFFERENT rivers — spatial extrapolation problem',
                 fontsize=13)
    ax.legend(fontsize=11, loc='lower left')
    ax.grid(True, alpha=0.3)
    
    # Add text annotation
    ax.text(0.98, 0.02, f'Train: {len(train_stations)} stations, {len(train_merged)} samples\n'
                         f'Val: {len(val_stations)} stations, {len(val_merged)} samples',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig_00_station_map.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: fig_00_station_map.png')

## 📊 FIGURE 2: Target Distributions

In [ ]:
n_targets = len(TARGET_COLS)
fig, axes = plt.subplots(2, n_targets, figsize=(6*n_targets, 10))
if n_targets == 1:
    axes = axes.reshape(2, 1)

colors = ['#2196F3', '#4CAF50', '#FF9800']

for i, col in enumerate(TARGET_COLS):
    short_name = col.split('(')[0].strip() if '(' in col else col[:20]
    
    # Row 1: Histogram
    ax = axes[0, i]
    ax.hist(train_merged[col].dropna(), bins=50, color=colors[i % 3], alpha=0.7, edgecolor='white')
    ax.axvline(train_merged[col].median(), color='red', linestyle='--', linewidth=2, label=f'median={train_merged[col].median():.1f}')
    ax.axvline(train_merged[col].mean(), color='black', linestyle=':', linewidth=2, label=f'mean={train_merged[col].mean():.1f}')
    ax.set_title(short_name, fontsize=12)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    # Add skewness annotation
    skew = train_merged[col].skew()
    ax.text(0.95, 0.95, f'skew={skew:.2f}', transform=ax.transAxes, ha='right', va='top',
            fontsize=10, bbox=dict(facecolor='white', alpha=0.8))
    
    # Row 2: Box plot
    ax2 = axes[1, i]
    ax2.boxplot(train_merged[col].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor=colors[i % 3], alpha=0.5))
    ax2.set_ylabel('Value')
    ax2.set_title(f'{short_name} — Outliers', fontsize=11)
    
    q1, q3 = train_merged[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    outliers = ((train_merged[col] < q1 - 1.5*iqr) | (train_merged[col] > q3 + 1.5*iqr)).sum()
    ax2.text(0.95, 0.95, f'outliers={outliers}', transform=ax2.transAxes, ha='right', va='top',
            fontsize=10, bbox=dict(facecolor='white', alpha=0.8))

fig.suptitle('📊 Target Variable Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_00_target_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_00_target_distributions.png')

## 📊 FIGURE 3: Samples per Station & Temporal Coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Samples per station (sorted)
if STATION_COL:
    station_counts = train_merged[STATION_COL].value_counts().sort_values(ascending=True)
    ax = axes[0]
    ax.barh(range(len(station_counts)), station_counts.values, color='#2196F3', alpha=0.7)
    ax.set_xlabel('Number of Samples')
    ax.set_ylabel(f'Station Index (of {len(station_counts)})')
    ax.set_title(f'Samples per Station\nmin={station_counts.min()}, max={station_counts.max()}, median={station_counts.median():.0f}')
    ax.axvline(station_counts.median(), color='red', linestyle='--', label='median')
    ax.legend()

# Right: Temporal distribution
if DATE_COL:
    ax2 = axes[1]
    train_merged[DATE_COL].dt.to_period('M').value_counts().sort_index().plot(kind='bar', ax=ax2, color='#4CAF50', alpha=0.7)
    ax2.set_title('Samples per Month')
    ax2.set_xlabel('Month')
    ax2.set_ylabel('Count')
    ax2.tick_params(axis='x', rotation=45)

plt.suptitle('📈 Data Coverage', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_00_data_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_00_data_coverage.png')

## 📊 FIGURE 4: Missing Values Heatmap

In [ ]:
null_pcts = train_merged.isnull().mean()
null_pcts_nonzero = null_pcts[null_pcts > 0].sort_values(ascending=False)

if len(null_pcts_nonzero) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(null_pcts_nonzero) * 0.3)))
    bars = ax.barh(range(len(null_pcts_nonzero)), null_pcts_nonzero.values * 100, 
                   color=['#FF5722' if v > 0.5 else '#FF9800' if v > 0.1 else '#4CAF50' for v in null_pcts_nonzero.values])
    ax.set_yticks(range(len(null_pcts_nonzero)))
    ax.set_yticklabels(null_pcts_nonzero.index, fontsize=9)
    ax.set_xlabel('% Missing')
    ax.set_title('🔍 Missing Values by Feature', fontweight='bold')
    
    # Color legend
    legend_elements = [mpatches.Patch(facecolor='#FF5722', label='>50% missing'),
                       mpatches.Patch(facecolor='#FF9800', label='10-50% missing'),
                       mpatches.Patch(facecolor='#4CAF50', label='<10% missing')]
    ax.legend(handles=legend_elements, loc='lower right')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig_00_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: fig_00_missing_values.png')
else:
    print('✅ No missing values in training data')

print(f'\nTotal columns: {len(train_merged.columns)}')
print(f'Columns with nulls: {len(null_pcts_nonzero)}')

## 📊 FIGURE 5: Spatial Distribution of Each Target

In [ ]:
if LAT_COL and LON_COL:
    station_means = train_merged.groupby([STATION_COL, LAT_COL, LON_COL])[TARGET_COLS].mean().reset_index()
    
    fig, axes = plt.subplots(1, n_targets, figsize=(7*n_targets, 7))
    if n_targets == 1:
        axes = [axes]
    
    for i, col in enumerate(TARGET_COLS):
        ax = axes[i]
        short_name = col.split('(')[0].strip() if '(' in col else col[:20]
        
        sc = ax.scatter(station_means[LON_COL], station_means[LAT_COL],
                       c=station_means[col], cmap='RdYlGn_r', s=60, alpha=0.8,
                       edgecolors='gray', linewidths=0.5)
        plt.colorbar(sc, ax=ax, shrink=0.8, label=col.split('(')[-1].replace(')', '') if '(' in col else '')
        
        # Add validation stations (unknown values)
        if STATION_COL in val_merged.columns:
            v_locs = val_merged.groupby(STATION_COL).agg({LAT_COL: 'first', LON_COL: 'first'}).reset_index()
        else:
            v_locs = val_merged[[LAT_COL, LON_COL]].drop_duplicates()
        ax.scatter(v_locs[LON_COL], v_locs[LAT_COL], c='black', marker='x', s=80, linewidths=2,
                  label='Validation (predict these)', zorder=5)
        
        ax.set_title(short_name, fontsize=12)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.legend(fontsize=9, loc='lower left')
        ax.set_xlim(16, 33)
        ax.set_ylim(-35, -22)
    
    fig.suptitle('🗺️ Mean Water Quality per Station (colored) + Validation Locations (×)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig_00_spatial_targets.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: fig_00_spatial_targets.png')

## 6. Save Base Datasets

In [ ]:
# Save as parquet
train_merged.to_parquet(f'{OUTPUT_DIR}/train_base.parquet', index=False)
val_merged.to_parquet(f'{OUTPUT_DIR}/val_base.parquet', index=False)

print(f'✅ Saved train_base.parquet: {train_merged.shape}')
print(f'✅ Saved val_base.parquet: {val_merged.shape}')
print(f'\n=== KEY INFO FOR NEXT NOTEBOOKS ===')
print(f'Target columns: {TARGET_COLS}')
print(f'Station column: {STATION_COL}')
print(f'Date column: {DATE_COL}')
print(f'Lat/Lon: {LAT_COL}, {LON_COL}')
print(f'Merge keys: {MERGE_KEYS}')

---
## Summary

| Metric | Value |
|--------|-------|
| Training samples | see shape above |
| Validation samples | see shape above |
| Training stations | see count above |
| Validation stations | see count above |
| Station overlap | should be 0 |
| Features from Landsat | see merge output |
| Features from TerraClimate | see merge output |

### Figures Produced
1. `fig_00_station_map.png` — Where are the stations? Train vs val locations
2. `fig_00_target_distributions.png` — What do the targets look like? Skewness?
3. `fig_00_data_coverage.png` — How many samples per station? Temporal gaps?
4. `fig_00_missing_values.png` — What's missing?
5. `fig_00_spatial_targets.png` — Spatial patterns in water quality